# __MODEL_LABEL_MARKDOWN__: champion and challenger review

Compare the current champion with published candidates and weekly challengers.
Select one explicit package version, review its SQL summary and metrics, then
promote that reviewed package in the final cell. Review needs no local model files.


In [ ]:
DATABASE_MODE = __DATABASE_MODE_LITERAL__  # "local" or "remote"
RUNTIME_MODULE = __RUNTIME_MODULE_LITERAL__  # e.g. "project_runtime.database"; never put secrets here
EXPECTED_REMOTE_DATABASE = __EXPECTED_REMOTE_DATABASE_LITERAL__
ALLOW_REMOTE_WRITES = False

MODEL_NAME = "__MODEL_NAME__"  # Set to None to select by label only.
MODEL_LABEL = "__MODEL_LABEL__"
DEPLOYMENT_SLOT = "__DEPLOYMENT_SLOT__"
PACKAGE_VERSION = None  # Set an explicit published package version to review.

reviewed = None


In [ ]:
from pathlib import Path

PROJECT_ROOT = next(
    candidate
    for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
    if (candidate / "pyproject.toml").is_file()
    and (candidate / "pricing_models").is_dir()
)

from pricing_pipeline.notebook import (
    connect,
    deploy_model_version,
    list_challengers,
    list_model_versions,
    load_registered_model,
    review_model_version,
)

MODEL_DIR = PROJECT_ROOT / "pricing_models/__PACKAGE_NAME__"


## List the champion and challengers

The table shows each saved fit's role, definition revision and refit type.
Weekly refits keep the same definition revision. Each fit has its own package
number. Choose that number in `PACKAGE_VERSION` to review and promote it.

`CHAMPION` is the package active in `DEPLOYMENT_SLOT`. `FORMER_CHAMPION` marks a
previous deployment. SQL users can read the same information in
`pricing.V_MODEL_REGISTRY`.


In [ ]:
pricing = connect(
    mode=DATABASE_MODE,
    runtime_module=RUNTIME_MODULE,
    local_root=MODEL_DIR / ".local",
    expected_remote_database=EXPECTED_REMOTE_DATABASE,
    allow_remote_writes=ALLOW_REMOTE_WRITES,
)
display(pricing.destination)


In [ ]:
model = load_registered_model(
    pricing,
    model_name=MODEL_NAME,
    model_label=MODEL_LABEL,
    deployment_slot=DEPLOYMENT_SLOT,
    source_root=MODEL_DIR,
)
versions = list_model_versions(pricing, model=model, technical=True)
deployable = versions.loc[
    versions["package_status"].astype(str).str.upper().eq("PUBLISHED")
].copy()
package_columns = [
    "package_version",
    "rate_package_id",
    "model_version",
    "model_kind",
    "data_as_of_date",
    "manifest_id",
    "current_rate_package_id",
]
current_champion = deployable.loc[
    deployable["rate_package_id"].eq(deployable["current_rate_package_id"])
]
print(f"Current champion in {DEPLOYMENT_SLOT}")
if current_champion.empty:
    print("No package is deployed in this slot.")
display(current_champion.loc[:, package_columns])
print("Published packages available for review")
display(deployable.loc[:, package_columns])
challengers = list_challengers(pricing, model=model)
print("Weekly challengers and their champion comparisons")
display(challengers)


## Select and review one package from SQL

Set `PACKAGE_VERSION` in the settings cell, then rerun this cell. No package is
selected automatically. Review the summary and metrics before describing the
promotion decision in the separate `DEPLOYMENT_REASON` cell below.

The review records the current champion. If that champion changes before promotion,
the database rejects the stale review. Refresh the lists and review again before retrying.


In [ ]:
reviewed = None
if PACKAGE_VERSION is None:
    raise ValueError(
        "Set PACKAGE_VERSION to an explicit published version, then rerun this review cell."
    )
if isinstance(PACKAGE_VERSION, bool) or not isinstance(PACKAGE_VERSION, int):
    raise ValueError("PACKAGE_VERSION must be an integer from the displayed published list.")
if deployable.empty:
    raise LookupError("No published candidate packages were found.")
if PACKAGE_VERSION not in set(deployable["package_version"].astype(int)):
    raise ValueError("PACKAGE_VERSION is not in the displayed published list.")
reviewed = review_model_version(
    pricing,
    model=model,
    package_version=PACKAGE_VERSION,
)
display(reviewed.summary)
display(reviewed.metrics)


## Promote the reviewed package

Enter the decision in the next cell after reviewing the selected package.
Then run the promotion cell. Editing this decision preserves the reviewed package.
Promotion changes the champion in `DEPLOYMENT_SLOT`. It does not rerun a fit.
Notebook 07 and scheduled monitoring create challengers without promoting them.
The expected-champion check prevents this cell from replacing a champion that changed
since review. Changing `PACKAGE_VERSION` also requires another review.


In [ ]:
DEPLOYMENT_REASON = ""  # Explain why this reviewed package should become the champion.


In [ ]:
try:
    if reviewed is None:
        raise ValueError("Run the SQL review cell before promotion.")
    if PACKAGE_VERSION != reviewed.package_version:
        raise ValueError(
            "PACKAGE_VERSION changed after review. Rerun the review cell before promotion."
        )
    if not DEPLOYMENT_REASON.strip():
        raise ValueError("Describe the approval for changing the slot's champion.")
    deployment = deploy_model_version(
        pricing,
        package=reviewed,
        reason=DEPLOYMENT_REASON,
    )
    display(deployment)
finally:
    pricing.engine.dispose()
